# Hyperparameter Tuning
This notebook covers how to tune the hyperparameters of random forests and histogram gradient-boosting.

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data, target = fetch_california_housing(return_X_y=True, as_frame=True)
target *= 100
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=0
)

## Random Forest
The main parameter in random forests is `n_estimators`. Generally, the more trees in the forest, the better the generalization performance. However, adding more trees slows down the fitting and prediction time.

We can also tune `max_features`, which controls the size of the random subset of features to consider when looking for the best split when growing trees. Smaller values lead to more random trees and more uncorrelated prediction errors, but too small a value may be too random. When `max_features` is `None`, it is equivalent to `max_features = n_features` which means the *only source of randomness comes from bagging*.

The two parameters that control the depth of the trees are `max_depth` and `max_leaf_nodes`. `max_depth` enforces symmetric trees, and `max_leaf_nodes` does not.

`min_samples_leaf` controls the minimum number of samples required to be at a leaf node. This means that a split point is only done if it leaves at least `min_samples_leaf` training samples in each of the left and right branches. Smaller samples promotes overfitting, larger samples can cause underfiting.

In [2]:
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

param_distributions = {
    "max_features": [1, 2, 3, 5, None],
    "max_leaf_nodes": [10, 100, 1000, None],
    "min_samples_leaf": [1, 2, 5, 10, 20, 50, 100],
}
search_cv = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=2),
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_error",
    n_iter=10,
    random_state=0,
    n_jobs=2,
)
search_cv.fit(data_train, target_train)

columns = [f"param_{name}" for name in param_distributions.keys()]
columns += ["mean_test_error", "std_test_error"]
cv_results = pd.DataFrame(search_cv.cv_results_)
cv_results["mean_test_error"] = -cv_results["mean_test_score"]
cv_results["std_test_error"] = cv_results["std_test_score"]
cv_results[columns].sort_values(by="mean_test_error")

,param_max_features,param_max_leaf_nodes,param_min_samples_leaf,mean_test_error,std_test_error
3,2,None,2,34.032926,0.647993
0,2,1000,10,36.906199,0.612488
7,None,None,20,37.313388,0.495720
4,5,100,2,40.072855,0.609909
8,None,100,10,40.453095,0.547283
6,None,1000,50,40.859307,0.524572
2,1,100,1,49.958593,0.958111
9,1,100,2,50.204552,0.727276
5,1,None,100,54.861733,0.601656
1,3,10,10,54.943305,0.649179


In [3]:
error = -search_cv.score(data_test, target_test)
print(
    f"On average, our random forest regressor makes an error of {error:.2f} k$"
)

On average, our random forest regressor makes an error of 33.74 k$


## Histogram Gradient-Boosting
For gradient-boosting, hyperparameters are coupled, so they cannot be set one after another.

`max_iter` controls the number of trees in the estimator. This differs from `n_estimators` in that it sets a *maximum*, the actual number of trees learned could be less.

`max_depth` and `max_leaf_nodes` can also be used on these trees, however, we tend to want shallow trees for gradient-boosting.

Finally, the `learning_rate` controls how much each correction contributes to the final prediction. Smaller learning rates means the corrections of a new tree result in small adjustments to the model prediction.

In [4]:
from scipy.stats import loguniform
from sklearn.ensemble import HistGradientBoostingRegressor

param_distributions = {
    "max_iter": [3, 10, 30, 100, 300, 1000],
    "max_leaf_nodes": [2, 5, 10, 20, 50, 100],
    "learning_rate": loguniform(0.01, 1),
}
search_cv = RandomizedSearchCV(
    HistGradientBoostingRegressor(),
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_error",
    n_iter=20,
    random_state=0,
    n_jobs=2,
)
search_cv.fit(data_train, target_train)

columns = [f"param_{name}" for name in param_distributions.keys()]
columns += ["mean_test_error", "std_test_error"]
cv_results = pd.DataFrame(search_cv.cv_results_)
cv_results["mean_test_error"] = -cv_results["mean_test_score"]
cv_results["std_test_error"] = cv_results["std_test_score"]
cv_results[columns].sort_values(by="mean_test_error")

,param_max_iter,param_max_leaf_nodes,param_learning_rate,mean_test_error,std_test_error
14,300,100,0.018640,31.067031,0.213683
6,300,20,0.047293,31.855584,0.172673
2,30,50,0.176656,32.613058,0.326207
13,300,10,0.297739,32.786227,0.531524
9,100,20,0.083745,33.055600,0.282339
19,100,10,0.215543,33.351342,0.208887
12,100,20,0.067503,33.690628,0.263127
16,300,5,0.059290,35.769657,0.486373
1,100,5,0.160519,36.314952,0.519693
0,1000,2,0.125207,40.758616,0.487892


In [5]:
error = -search_cv.score(data_test, target_test)
print(f"On average, our HGBT regressor makes an error of {error:.2f} k$")

On average, our HGBT regressor makes an error of 30.66 k$
